In [2]:
import sys
import os
import time
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

sys.path.append(os.path.abspath('..'))

from src.preprocessing import preparar_dados_limpos
from src.train_models import extrair_features_e_dividir, salvar_modelo_e_vetorizador
from src.evaluate import avaliar_modelo

caminho = r'..\data\raw\amostra_processos_classes_recorrentes_27022025.csv'
df = preparar_dados_limpos(caminho)

X_treino, X_teste, y_treino, y_teste, vetorizador = extrair_features_e_dividir(df)

In [3]:
resultados_modelos = []

def registrar_metricas(nome_modelo, y_true, y_pred, tempo_treino):
    acc = accuracy_score(y_true, y_pred)
    _, _, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    resultados_modelos.append({
        "Modelo": nome_modelo,
        "Acurácia": round(acc, 4),
        "F1-Score": round(f1, 4),
        "Tempo (s)": round(tempo_treino, 2)
    })

In [4]:
inicio_nb = time.time()
modelo_nb = MultinomialNB()
modelo_nb.fit(X_treino, y_treino)
tempo_nb = time.time() - inicio_nb

y_pred_nb = modelo_nb.predict(X_teste)

avaliar_modelo(modelo_nb, X_teste, y_teste, "Naive Bayes")
salvar_modelo_e_vetorizador(modelo_nb, vetorizador, "modelo_naive_bayes")
registrar_metricas("Naive Bayes", y_teste, y_pred_nb, tempo_nb)


RELATÓRIO DE CLASSIFICAÇÃO - NAIVE BAYES
                                                                                                                                                                                                                                          precision    recall  f1-score   support

                                                                                                                                                                                                           Agravo de Instrumento ( CPC )       0.58      0.81      0.68       400
                                                                                                                                                                                                                  Carta Precatória Cível       0.77      0.82      0.79       800
                                                                                                                                       

In [5]:
inicio_rf = time.time()
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
modelo_rf.fit(X_treino, y_treino)
tempo_rf = time.time() - inicio_rf

y_pred_rf = modelo_rf.predict(X_teste)

avaliar_modelo(modelo_rf, X_teste, y_teste, "Random Forest")
salvar_modelo_e_vetorizador(modelo_rf, vetorizador, "modelo_random_forest")
registrar_metricas("Random Forest", y_teste, y_pred_rf, tempo_rf)


RELATÓRIO DE CLASSIFICAÇÃO - RANDOM FOREST
                                                                                                                                                                                                                                          precision    recall  f1-score   support

                                                                                                                                                                                                           Agravo de Instrumento ( CPC )       0.67      0.93      0.77       400
                                                                                                                                                                                                                  Carta Precatória Cível       0.90      0.91      0.90       800
                                                                                                                                     

In [6]:
from sklearn.linear_model import LogisticRegression

inicio_lr = time.time()
modelo_lr = LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42)
modelo_lr.fit(X_treino, y_treino)
tempo_lr = time.time() - inicio_lr

y_pred_lr = modelo_lr.predict(X_teste)

avaliar_modelo(modelo_lr, X_teste, y_teste, "Regressao Logistica")
salvar_modelo_e_vetorizador(modelo_lr, vetorizador, "modelo_regressao_logistica")
registrar_metricas("Regressão Logística", y_teste, y_pred_lr, tempo_lr)


RELATÓRIO DE CLASSIFICAÇÃO - REGRESSAO LOGISTICA
                                                                                                                                                                                                                                          precision    recall  f1-score   support

                                                                                                                                                                                                           Agravo de Instrumento ( CPC )       0.73      0.88      0.80       400
                                                                                                                                                                                                                  Carta Precatória Cível       0.90      0.89      0.89       800
                                                                                                                               

In [7]:
df_comparacao = pd.DataFrame(resultados_modelos)
display(df_comparacao.sort_values(by="F1-Score", ascending=False))

,Modelo,Acurácia,F1-Score,Tempo (s)
1,Random Forest,0.8527,0.8435,112.14
2,Regressão Logística,0.8276,0.8227,193.29
0,Naive Bayes,0.7384,0.7199,1.53
